# Online Recruitment Fraud Detection - Bidirectional LSTM Baseline
This notebook implements an optimized Bidirectional LSTM (Bi-LSTM) model using PyTorch.

### State-of-the-Art Enhancements Included:
1. **2-Layer Deep Bi-LSTM**: Captures hierarchical word relationships over a 1-layer model.
2. **AdamW Optimizer**: Better weight decay implementation for deep network regularization.
3. **Cosine Annealing Scheduler**: Dynamically scales learning rate over epochs to search local minima.
4. **Early Stopping & Gradient Clipping**: Prevents overfitting and guards against gradient explosions.
5. **Structured Concatenation**: Leverages structured text concatenation across multiple job post fields.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

csv_path = "fake_job_postings.csv"

if not os.path.exists(csv_path):
    try:
        from google.colab import files
        print("Dataset 'fake_job_postings.csv' not found. Please upload it:")
        uploaded = files.upload()
    except ImportError:
        print(f"Local file '{csv_path}' not found. Please place it in the same directory.")
else:
    print(f"Dataset found at '{csv_path}'. Skipping upload prompt.")

## Preprocessing & Vocabulary Fitting

In [ ]:
df = pd.read_csv(csv_path)
df.fillna(" ", inplace=True)

# Structured Concatenation (10 fields)
df['combined_text'] = (
    "Title: " + df['title'].astype(str) + 
    " | Company Profile: " + df['company_profile'].astype(str) + 
    " | Description: " + df['description'].astype(str) + 
    " | Requirements: " + df['requirements'].astype(str) + 
    " | Benefits: " + df['benefits'].astype(str) + 
    " | Employment Type: " + df['employment_type'].astype(str) + 
    " | Required Experience: " + df['required_experience'].astype(str) + 
    " | Required Education: " + df['required_education'].astype(str) + 
    " | Industry: " + df['industry'].astype(str) + 
    " | Function: " + df['function'].astype(str)
)

# Split train, val, test
train_df, test_df = train_test_split(df, test_size=0.20, stratify=df['fraudulent'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.125, stratify=train_df['fraudulent'], random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Build Vocabulary cleanly
words = []
for text in train_df['combined_text']:
    words.extend(text.lower().split())
    
word_counts = Counter(words)
# Keep words that occur at least twice
filtered_words = [word for word, count in word_counts.items() if count >= 2]
vocab = {word: idx + 2 for idx, word in enumerate(filtered_words)}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1
print(f"Vocabulary size: {len(vocab)}")

## PyTorch Dataset & DataLoader

In [ ]:
def text_to_indices(text, vocab, max_len=512):
    tokens = text.lower().split()[:max_len]
    indices = [vocab.get(token, 1) for token in tokens]
    if len(indices) < max_len:
        indices += [0] * (max_len - len(indices))
    return indices

class TextDataset(Dataset):
    def __init__(self, df, vocab, max_len=512):
        self.labels = df['fraudulent'].values
        self.features = [text_to_indices(text, vocab, max_len) for text in df['combined_text']]
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        return torch.tensor(self.features[idx]), torch.tensor(self.labels[idx], dtype=torch.float32)

train_dataset = TextDataset(train_df, vocab)
val_dataset = TextDataset(val_df, vocab)
test_dataset = TextDataset(test_df, vocab)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## Define 2-Layer Bi-LSTM Architecture

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128, output_dim=1, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            bidirectional=True, 
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim)
        )
        
    def forward(self, text):
        embedded = self.embedding(text)
        lstm_out, _ = self.lstm(embedded)
        # Global Max Pooling over sequence length
        pool_out, _ = torch.max(lstm_out, dim=1)
        logits = self.fc(pool_out)
        return logits.squeeze(1)

## Model Training (AdamW, Schedulers, and Early Stopping)

In [ ]:
model = LSTMClassifier(vocab_size=len(vocab)).to(device)

pos_weight = torch.tensor([2.0]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# AdamW optimizer with Cosine Annealing Learning Rate scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

best_val_f1 = 0.0
epochs = 10
patience = 3
patience_counter = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        
    scheduler.step()
    
    # Validate
    model.eval()
    val_preds = []
    val_labels = []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = torch.sigmoid(model(inputs))
            val_preds.extend(outputs.cpu().numpy())
            val_labels.extend(labels.numpy())
            
    val_labels_int = [int(l) for l in val_labels]
    val_preds_bin = (np.array(val_preds) >= 0.5).astype(int)
    report = classification_report(val_labels_int, val_preds_bin, output_dict=True, zero_division=0)
    val_f1 = report.get('1', report.get('1.0', report.get(1, {}))).get('f1-score', 0.0)
    
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")
    
    # Early Stopping check
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "lstm_best_model.pt")
        print("  Saved new best model checkpoint!")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

## Evaluate on Test Set

In [ ]:
# Load best model weights
model.load_state_dict(torch.load("lstm_best_model.pt"))
model.eval()

test_probs = []
test_labels = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = torch.sigmoid(model(inputs))
        test_probs.extend(outputs.cpu().numpy())
        test_labels.extend(labels.numpy())

test_preds = (np.array(test_probs) >= 0.5).astype(int)
print("Bi-LSTM Classification Report on Test Set:")
print(classification_report(test_labels, test_preds))

# Formatted metrics summary
test_acc = accuracy_score(test_labels, test_preds)
test_prec = precision_score(test_labels, test_preds, zero_division=0)
test_rec = recall_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, test_preds)
test_auc = roc_auc_score(test_labels, test_probs)

print("\n--- Formatted Metrics Summary ---")
print(f"Accuracy:  {test_acc*100:.2f}%")
print(f"Precision: {test_prec*100:.2f}%")
print(f"Recall:    {test_rec*100:.2f}%")
print(f"F1-Score:  {test_f1*100:.2f}%")
print(f"ROC-AUC:   {test_auc*100:.2f}%")

# Plot Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(test_labels, test_preds), annot=True, fmt='d', cmap='Greens')
plt.title('Bi-LSTM Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()